<div style="background-color:#EAEAEA;padding:20px;border-left:5px solid #6C757D;border-radius:6px;">
<table style="width:100%; border:none;"><tr style="border:none;">
<td style="border:none; vertical-align:top;">
<h1 style="font-size:32px; margin-top:0;">Master's Thesis</h1>
<hr style="margin:16px 0 22px 0;">
<p style="font-size:22px; line-height:1.5; margin:0;"><strong>Master's Degree in Advanced Physics</strong> - <strong>Universitat de Valencia</strong></p>
<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:</p>
<div style="font-size:25px;font-weight:700;line-height:1.3;margin-top:14px;margin-bottom:26px;">Fast Simulation of Neutrino Oscillations in Matter</div>
<p style="font-size:14px; line-height:1.55;"><strong>Author</strong><br>Juan Ramon Diaz Santos - <a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a></p>
<p style="font-size:14px; line-height:1.55;"><strong>Supervisors</strong><br>Roberto Ruiz de Austri Bazan - <a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>Michele Lucente - <a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a></p>
<p style="font-size:14px; line-height:1.55; margin-bottom:0;"><strong>Date</strong><br>September 2026</p></td>
<td style="border:none;width:230px;padding-left:25px;text-align:right;vertical-align:top;"><img src="../../logo_uv.png" alt="Universitat de Valencia" style="width:200px; margin-top:5px;"></td>
</tr></table></div>

# Intrinsic Validation 5 - Perturbative Method
---
This notebook collects the analytical-method studies for the Standard Model, NSI and $3+1$ sterile-neutrino extension.

## Table of Contents

| # | Section |
|---|---|
| [0](#0.-Theoretical-Framework) | **Theoretical Framework** |
| [1](#1.-Libraries) | **Libraries** |
| [2](#2.-Paths-and-Configuration) | **Paths and Configuration** |
| [3](#3.-Standard-Model) | **Standard Model** |
| [4](#4.-NSI) | **NSI** |
| [5](#5.-$3+1$-Sterile-Neutrino) | **$3+1$ Sterile Neutrino** |
| [6](#6.-Summary) | **Summary** |


## 0. Theoretical Framework
---
This notebook isolates analytical propagation options. In Earth, each segment is written as $H=H_0+H_1(x)$ and the evolutor is truncated after the first Dyson correction. `reunitarize=True` projects that approximation back onto the unitary group. `analytic_eigenvalues=True` evaluates the reduced Hamiltonian spectrum with Cardano for three flavours and Ferrari for four flavours instead of `torch.linalg.eigvalsh`.

For the Standard-Model Sun, the local two-level Landau-Zener option is also compared. It is not available for NSI or sterile `adiabatic_exact`, whose Hamiltonians require full pointwise diagonalization.

**References**

No external references are cited in this notebook.


## 1. Libraries


In [ ]:
from __future__ import annotations
%matplotlib inline
import dataclasses
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tpeanuts.notebooks.notebookConfig import load_notebook_config
from tpeanuts.notebooks.notebooks_helper import ABSOLUTE_THRESHOLD, save_and_show, to_numpy, validation_metrics, validation_summary_plot
from tpeanuts.config.propagation import PropagationConfig
from tpeanuts.medium.earth.profile import EarthParameters, EarthProfile
from tpeanuts.medium.earth.probability import earth_probability_state
from tpeanuts.medium.solar.profile import SolarProfile
from tpeanuts.medium.solar.probability import solar_probability_mass
from tpeanuts.util.context import RuntimeContext


## 2. Paths and Configuration

### 2.1 Paths

`load_notebook_config()` resolves the package directory and the common Torch, NumPy and plotting configuration. Generated figures are written under `validation/intrinsic/`.


In [ ]:
config = load_notebook_config()
DEVICE, DTYPE = config.device, config.dtype
context = RuntimeContext.resolve(DEVICE, DTYPE)
OUTPUT_DIR = config.output_dir('validation', 'intrinsic')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Package dir : {config.package_dir}')
print(f'Output dir  : {OUTPUT_DIR}')
print(f'Device      : {context.device}   dtype: {context.dtype}')


### 2.2 Shared Configuration

The three model sections use identical Earth, energy, angular, depth, precision and error-threshold settings. Only the oscillation/BSM model is changed.


In [ ]:
SM_PRESET = '_SM_NUFIT52_NO'
NSI_PRESET = 'nsi_globalfit_esteban2018'
STERILE_PRESET = 'sterile_3p1_bestfit_giunti2017'
MODEL_LABELS = {'sm': 'Standard Model', 'nsi': 'NSI', 'sterile': '3+1 sterile'}
DEPTH_M, N_EARTH_STEPS = 1000.0, 400
LEGACY_PRECISION = False
ABSOLUTE_THRESHOLD = 1e-3
earth = EarthProfile(params=EarthParameters(density_provider='prem', profile_perturbative_name='even_power', profile_perturbative_kwargs={'density_file': str(config.earth_density_file), 'tabulated_density': False}), context=context)
solar = SolarProfile.default(context=context)
def model_oscillation(model, runtime_context=context):
    if model == 'sm':
        return PropagationConfig.oscillation_parameters_from_preset(SM_PRESET, context=runtime_context, antinu=False)
    if model == 'nsi':
        return PropagationConfig.oscillation_parameters_from_preset(SM_PRESET, context=runtime_context, antinu=False, NSI_extension=NSI_PRESET)
    if model == 'sterile':
        return PropagationConfig.oscillation_parameters_from_preset(STERILE_PRESET, context=runtime_context, antinu=False)
    raise ValueError(f'Unknown model: {model!r}')
def select_model(model):
    global oscillation, N_FLAVOURS, CDTYPE, BASIS, FLAVOUR_LABELS, FLAVOUR_COLOURS
    oscillation = model_oscillation(model)
    N_FLAVOURS = int(oscillation.pmns.n_flavours)
    CDTYPE = torch.complex128 if context.dtype == torch.float64 else torch.complex64
    BASIS = torch.eye(N_FLAVOURS, device=context.device, dtype=CDTYPE)
    FLAVOUR_LABELS = [r'$\nu_e$', r'$\nu_\mu$', r'$\nu_\tau$'] + ([r'$\nu_s$'] if N_FLAVOURS == 4 else [])
    FLAVOUR_COLOURS = tuple(f'C{i}' for i in range(N_FLAVOURS))
    print(f'{MODEL_LABELS[model]}: {N_FLAVOURS} flavours, preset={oscillation.preset_name}')


### 2.3 Shared Study Helpers

The helpers below centralize the repeated scans and plotting logic while each model section explicitly selects its own oscillation parameters and flavour dimension.


In [ ]:
PERTURBATIVE_SUMMARIES = []
E_STUDY = torch.logspace(np.log10(100.0), np.log10(2e4), 60, device=context.device, dtype=context.dtype)
ETA_STUDY = torch.linspace(0.05, 1.50, 60, device=context.device, dtype=context.dtype)
def earth_two_scans(method, reunitarize, analytic_eigenvalues=False):
    state = BASIS[0]
    energy_scan = earth_probability_state(state, earth, oscillation, E_STUDY[:,None], torch.tensor(0.60, device=context.device, dtype=context.dtype), DEPTH_M, method=method, massbasis=False, nsteps=N_EARTH_STEPS, context=context, reunitarize=reunitarize, analytic_eigenvalues=analytic_eigenvalues, legacy_precision=LEGACY_PRECISION).reshape(E_STUDY.numel(),N_FLAVOURS)
    angle_scan = torch.stack([earth_probability_state(state, earth, oscillation, torch.tensor(5000.0, device=context.device, dtype=context.dtype), eta, DEPTH_M, method=method, massbasis=False, nsteps=N_EARTH_STEPS, context=context, reunitarize=reunitarize, analytic_eigenvalues=analytic_eigenvalues, legacy_precision=LEGACY_PRECISION) for eta in ETA_STUDY])
    return energy_scan, angle_scan
def reunitarization_study(model, prefix):
    select_model(model)
    reference=earth_two_scans('numerical',False); tested_false=earth_two_scans('analytical',False); tested_true=earth_two_scans('analytical',True)
    rows=[]
    for label,tested in [('reunitarize=False',tested_false),('reunitarize=True',tested_true)]:
        row,_,_=validation_metrics(label,torch.cat(reference),torch.cat(tested),reference_label='numerical',tested_label=label,absolute_threshold=ABSOLUTE_THRESHOLD); row['case']=f'{MODEL_LABELS[model]}: {label}'; rows.append(row); PERTURBATIVE_SUMMARIES.append(row)
    display(pd.DataFrame(rows).set_index('case').style.format('{:.3e}'))
    fig,axes=plt.subplots(3,2,figsize=(14,11),sharex='col')
    for column,(x,index,xlabel) in enumerate([(E_STUDY,0,'E [MeV]'),(ETA_STUDY,1,'eta [rad]')]):
        ref=reference[index]
        for flavour,(label,colour) in enumerate(zip(FLAVOUR_LABELS,FLAVOUR_COLOURS)):
            axes[0,column].plot(to_numpy(x),to_numpy(ref[:,flavour]),color=colour,label=f'{label} numerical'); axes[0,column].plot(to_numpy(x),to_numpy(tested_false[index][:,flavour]),'--',color=colour); axes[0,column].plot(to_numpy(x),to_numpy(tested_true[index][:,flavour]),':',color=colour)
        for tested,label,colour in [(tested_false[index],'False','C4'),(tested_true[index],'True','C5')]:
            _,absolute,relative=validation_metrics(label,ref,tested,absolute_threshold=ABSOLUTE_THRESHOLD); axes[1,column].plot(to_numpy(x),to_numpy(absolute.amax(dim=-1)),label=label,color=colour); axes[2,column].plot(to_numpy(x),to_numpy(relative.amax(dim=-1)),label=label,color=colour)
        axes[0,column].legend(fontsize=6,ncol=2); axes[1,column].set_yscale('log'); axes[2,column].set_yscale('log'); axes[2,column].set_xlabel(xlabel)
    for ax,label in zip(axes[:,0],['Probability','Maximum absolute error','Maximum relative error']): ax.set_ylabel(label)
    for ax in axes.flat: ax.grid(alpha=.25)
    fig.suptitle(f'{MODEL_LABELS[model]}: perturbative reunitarization'); fig.tight_layout(); save_and_show(f'{prefix}_reunitarization.png',fig,output_dir=OUTPUT_DIR,show_plots=config.show_plots)
def eigenvalue_study(model, prefix):
    select_model(model); reference=earth_two_scans('analytical',False,False); tested=earth_two_scans('analytical',False,True); solver='Ferrari' if N_FLAVOURS==4 else 'Cardano'
    row,_,_=validation_metrics(f'{solver} vs torch',torch.cat(reference),torch.cat(tested),reference_label='torch.linalg.eigvalsh',tested_label=solver,absolute_threshold=ABSOLUTE_THRESHOLD); row['case']=f'{MODEL_LABELS[model]}: {solver} vs torch'; PERTURBATIVE_SUMMARIES.append(row); display(pd.DataFrame([row]).set_index('case').style.format('{:.3e}'))
    fig,axes=plt.subplots(3,2,figsize=(14,11),sharex='col')
    for column,(x,index,xlabel) in enumerate([(E_STUDY,0,'E [MeV]'),(ETA_STUDY,1,'eta [rad]')]):
        ref=reference[index]; tst=tested[index]
        for flavour,(label,colour) in enumerate(zip(FLAVOUR_LABELS,FLAVOUR_COLOURS)):
            axes[0,column].plot(to_numpy(x),to_numpy(ref[:,flavour]),color=colour,label=f'{label} torch'); axes[0,column].plot(to_numpy(x),to_numpy(tst[:,flavour]),'--',color=colour,label=f'{label} {solver}')
        _,absolute,relative=validation_metrics(solver,ref,tst,absolute_threshold=ABSOLUTE_THRESHOLD); axes[1,column].plot(to_numpy(x),to_numpy(absolute.amax(dim=-1))); axes[2,column].plot(to_numpy(x),to_numpy(relative.amax(dim=-1)))
        axes[0,column].legend(fontsize=6,ncol=2); axes[1,column].set_yscale('log'); axes[2,column].set_yscale('log'); axes[2,column].set_xlabel(xlabel)
    for ax,label in zip(axes[:,0],['Probability','Maximum absolute error','Maximum relative error']): ax.set_ylabel(label)
    for ax in axes.flat: ax.grid(alpha=.25)
    fig.suptitle(f'{MODEL_LABELS[model]}: {solver} versus torch eigenvalues'); fig.tight_layout(); save_and_show(f'{prefix}_eigenvalues.png',fig,output_dir=OUTPUT_DIR,show_plots=config.show_plots)


## 3. Standard Model

This section tests the analytical Earth evolutor for the Standard Model model. Numerical propagation is the reference for reunitarization, while `torch.linalg.eigvalsh` is the reference for the closed-form spectrum.

### 3.1 Perturbative Reunitarization

A pure $\nu_e$ state is propagated through Earth over common energy and nadir-angle scans with `reunitarize=False` and `True`.

### 3.2 Analytical Eigenvalues

The same perturbative calculation compares numerical Hamiltonian eigenvalues against Cardano.

### 3.3 Solar Landau-Zener Option

The Standard-Model `adiabatic_approximated` calculation compares `use_LZ=False` and `True` first at the solar surface and then after analytical Earth propagation to a detector at $1000\,\mathrm{m}$ and $\eta=\pi/4$. Any detector-level difference must originate in the solar mass weights because the Earth transformation is common. This subsection is intentionally absent for NSI and sterile models.


In [ ]:
reunitarization_study('sm','intrinsic5_sm_fig31')
eigenvalue_study('sm','intrinsic5_sm_fig32')
solar_no_lz = dataclasses.replace(solar, use_LZ=False)
solar_with_lz = dataclasses.replace(solar, use_LZ=True)
E_lz = torch.linspace(0.5,15.0,120,device=context.device,dtype=context.dtype)
lz_reference = solar_probability_mass(oscillation,E_lz,solar_no_lz,'8B',method='adiabatic_approximated',legacy_precision=LEGACY_PRECISION)
lz_tested = solar_probability_mass(oscillation,E_lz,solar_with_lz,'8B',method='adiabatic_approximated',legacy_precision=LEGACY_PRECISION)
lz_surface_row,_,_=validation_metrics('solar surface: LZ True vs False',lz_reference,lz_tested,reference_label='LZ False',tested_label='LZ True',absolute_threshold=ABSOLUTE_THRESHOLD)
ETA_LZ_DETECTOR = torch.tensor(torch.pi/4,device=context.device,dtype=context.dtype)
lz_detector_reference = earth_probability_state(lz_reference,earth,oscillation,E_lz[:,None],ETA_LZ_DETECTOR,DEPTH_M,method='analytical',massbasis=True,nsteps=N_EARTH_STEPS,context=context,reunitarize=False,legacy_precision=LEGACY_PRECISION).reshape(E_lz.numel(),3)
lz_detector_tested = earth_probability_state(lz_tested,earth,oscillation,E_lz[:,None],ETA_LZ_DETECTOR,DEPTH_M,method='analytical',massbasis=True,nsteps=N_EARTH_STEPS,context=context,reunitarize=False,legacy_precision=LEGACY_PRECISION).reshape(E_lz.numel(),3)
lz_detector_row,_,_=validation_metrics('Earth detector: LZ True vs False',lz_detector_reference,lz_detector_tested,reference_label='LZ False',tested_label='LZ True',absolute_threshold=ABSOLUTE_THRESHOLD)
lz_surface_row['case']='Standard Model: LZ at solar surface'; lz_detector_row['case']='Standard Model: LZ at Earth detector'; PERTURBATIVE_SUMMARIES.extend([lz_surface_row,lz_detector_row])
display(pd.DataFrame([lz_surface_row,lz_detector_row]).set_index('case').style.format('{:.3e}'))


## 4. NSI

This section tests the analytical Earth evolutor for the NSI model. Numerical propagation is the reference for reunitarization, while `torch.linalg.eigvalsh` is the reference for the closed-form spectrum.

### 4.1 Perturbative Reunitarization

A pure $\nu_e$ state is propagated through Earth over common energy and nadir-angle scans with `reunitarize=False` and `True`.

### 4.2 Analytical Eigenvalues

The same perturbative calculation compares numerical Hamiltonian eigenvalues against Cardano.


In [ ]:
reunitarization_study('nsi','intrinsic5_nsi_fig41')
eigenvalue_study('nsi','intrinsic5_nsi_fig42')


## 5. $3+1$ Sterile Neutrino

This section tests the analytical Earth evolutor for the $3+1$ Sterile Neutrino model. Numerical propagation is the reference for reunitarization, while `torch.linalg.eigvalsh` is the reference for the closed-form spectrum.

### 5.1 Perturbative Reunitarization

A pure $\nu_e$ state is propagated through Earth over common energy and nadir-angle scans with `reunitarize=False` and `True`.

### 5.2 Analytical Eigenvalues

The same perturbative calculation compares numerical Hamiltonian eigenvalues against Ferrari.


In [ ]:
reunitarization_study('sterile','intrinsic5_sterile_fig51')
eigenvalue_study('sterile','intrinsic5_sterile_fig52')


## 6. Summary
This summary combines every analytical-option comparison: perturbative reunitarization, closed-form versus Torch eigenvalues, and the two Standard-Model Landau-Zener stages. Each row retains its own physically appropriate reference.

The table should therefore be interpreted by comparison type rather than as a ranking between models. The $2\times2$ chart provides a compact view of maximum and mean absolute and thresholded-relative discrepancies.


In [ ]:
perturbative_summary = pd.DataFrame(PERTURBATIVE_SUMMARIES).set_index('case')
display(perturbative_summary.style.format({column: '{:.3e}' for column in perturbative_summary.select_dtypes(include='number').columns}))
validation_summary_plot(PERTURBATIVE_SUMMARIES, title='Perturbative and adiabatic option summary', filename='intrinsic5_fig6_summary.png', output_dir=OUTPUT_DIR, show_plots=config.show_plots)
assert np.isfinite(perturbative_summary[['max_abs','mean_abs','max_rel','mean_rel']].to_numpy()).all()
